In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

In [ ]:
df = pd.read_csv("SN_m_tot_V2.0.csv",delimiter=';',decimal='.')

In [ ]:
df

In [ ]:
plt.figure(figsize=(16,4))


x = df["rok_norm"]          # or df.index
y = df["sunspot"]

# frac = fraction of data used for each local regression
smoothed = lowess(y, x, frac=0.01)

# smoothed is an array with columns [x, y_smoothed]
df["sunspot_lowess"] = smoothed[:, 1]

plt.scatter(x, y, alpha=0.4, label="Obserwacje",s=5)
plt.plot(x, df["sunspot_lowess"], color="red", label="Krzywa LOWESS")

plt.xlim(1749,1800)
plt.ylabel("International Sunspot Number")
plt.xlabel("Year")
plt.legend()
plt.grid(ls=":")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16,4))


x = df["rok_norm"]          # or df.index
y = df["sunspot"]

# frac = fraction of data used for each local regression
smoothed = lowess(y, x, frac=0.01)

# smoothed is an array with columns [x, y_smoothed]
df["sunspot_lowess"] = smoothed[:, 1]

# plt.scatter(x, y, alpha=0.4, label="Obserwacje",s=5)
plt.plot(x, df["sunspot_lowess"].diff(), color="red", label="Krzywa LOWESS")

plt.xlim(1950,2027)
plt.ylabel("International Sunspot Number")
plt.xlabel("Year")
plt.legend()
plt.grid(ls=":")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

x = df["rok_norm"]
y = df["sunspot"]

# 1. Compute LOWESS smoothing first
smoothed = lowess(y, x, frac=0.04)
df["sunspot_lowess"] = smoothed[:, 1]

# 2. Now compute sign changes on the smoothed curve's derivative
df["sld"] = df["sunspot_lowess"].diff()
signs = np.sign(df["sld"])
crossing_flags = (signs.diff().abs() == 2)

# 3. Cumulative count -> staircase pattern
df["crossings"] = crossing_flags.cumsum()
df["crossings"] = df["crossings"].bfill()  # handle leading NaNs so the line starts at 0, not blank

# Plot
plt.figure(figsize=(16, 4))
plt.plot(x, df["crossings"], color="red", label="Cumulative crossings (LOWESS turning points)")

plt.xlim(1950, 2027)
plt.ylabel("Cumulative sign changes")
plt.xlabel("Year")
plt.legend()
plt.grid(ls=":")
plt.tight_layout()
plt.show()

In [ ]:
# Diff the staircase to get spikes at each jump
df["crossings_diff"] = df["crossings"].diff()

# Spikes = where the diff is nonzero (i.e. a crossing happened)
spike_mask = df["crossings_diff"] > 0

# Get the rok_norm (x) values at each spike
spike_times = df.loc[spike_mask, "rok_norm"]

# Distances between consecutive spikes
spike_distances = spike_times.diff()

print(spike_times)
print(spike_distances)

In [ ]:
plt.figure(figsize=(16,4))
plt.stem(spike_times, np.ones(len(spike_times)), basefmt=" ")
plt.xlim(1950, 2027)
plt.xlabel("Year")
plt.ylabel("Crossing spike")
plt.title("Turning points of LOWESS curve")
plt.grid(ls=":")
plt.tight_layout()
plt.show()

In [ ]:
print(spike_distances.mean())